# Telegram File Link Bot - Google Colab Setup

This notebook installs and runs **telegram-file-link-bot** in Google Colab.

## What this notebook does
1. Ask for BOT TOKEN / API ID / API HASH / public domain / port
2. Install dependencies
3. Clone/setup project files
4. Create `.env`
5. Start the bot and print status URL

In [ ]:
# Step 1 - enter required credentials and runtime options
import getpass
import os

BOT_TOKEN = getpass.getpass("Telegram BOT TOKEN: ").strip()
API_ID = input("Telegram API ID: ").strip()
API_HASH = getpass.getpass("Telegram API HASH: ").strip()
PUBLIC_BASE_URL = input("Public domain URL (optional, e.g. https://example.com): ").strip()
PORT = (input("Port [8080]: ").strip() or "8080")

ADMIN_IDS = input("Admin IDs (comma separated, optional): ").strip()
LINK_EXPIRY_SECONDS = (input("Default link expiry in seconds [86400]: ").strip() or "86400")

if not BOT_TOKEN or not API_ID or not API_HASH:
    raise ValueError("BOT_TOKEN, API_ID, and API_HASH are required")

print("Credentials captured successfully.")

In [ ]:
# Step 2 - clone project and install dependencies automatically
import subprocess
from pathlib import Path

DEFAULT_REPO_URL = "https://github.com/kakarotoncloud/CR-F2L.git"
DEFAULT_BRANCH = "main"
PROJECT_DIR = Path("/content/telegram-file-link-bot")

if not PROJECT_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            DEFAULT_BRANCH,
            DEFAULT_REPO_URL,
            str(PROJECT_DIR),
        ],
        check=True,
    )
else:
    print(f"Project folder already exists: {PROJECT_DIR}")

print(f"Using project directory: {PROJECT_DIR}")
subprocess.run(["python", "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run(["python", "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)
print("Dependencies installed.")

In [ ]:
# Step 3 - write environment file
from pathlib import Path
import secrets

env_path = PROJECT_DIR / ".env"
secret = secrets.token_urlsafe(32)

env_lines = [
    f"BOT_TOKEN={BOT_TOKEN}",
    f"API_ID={API_ID}",
    f"API_HASH={API_HASH}",
    f"PUBLIC_BASE_URL={PUBLIC_BASE_URL if PUBLIC_BASE_URL else f'http://127.0.0.1:{PORT}'}",
    f"PORT={PORT}",
    "SERVER_HOST=0.0.0.0",
    f"ADMIN_IDS={ADMIN_IDS}",
    f"LINK_EXPIRY_SECONDS={LINK_EXPIRY_SECONDS}",
    f"LINK_SIGNING_SECRET={secret}",
    "RATE_LIMIT_REQUESTS=8",
    "RATE_LIMIT_WINDOW_SECONDS=60",
    "MAX_FILE_SIZE_MB=2048",
    "LOG_LEVEL=INFO",
    "FFMPEG_ENABLED=true",
    "DATABASE_PATH=data/bot.db",
    "STORAGE_PATH=storage/files",
    "HLS_PATH=storage/hls",
    "PYROGRAM_WORKDIR=.pyrogram",
]

env_path.write_text("\n".join(env_lines) + "\n", encoding="utf-8")
print(f"Created {env_path}")

In [ ]:
# Step 4 + Step 5 - start the bot and print status URL
import os
import signal
import subprocess
import threading
import time
from pathlib import Path

os.chdir(PROJECT_DIR)
print(f"Working directory: {Path.cwd()}")

if "bot_process" in globals() and bot_process.poll() is None:
    print("An existing bot process is running. Stopping it first...")
    bot_process.send_signal(signal.SIGTERM)
    time.sleep(2)

bot_process = subprocess.Popen(
    ["python", "-m", "bot.main"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

def _stream_logs(proc):
    for line in proc.stdout:
        print(line, end="")

log_thread = threading.Thread(target=_stream_logs, args=(bot_process,), daemon=True)
log_thread.start()

time.sleep(4)
status = "running" if bot_process.poll() is None else f"stopped (exit={bot_process.returncode})"
running_url = PUBLIC_BASE_URL if PUBLIC_BASE_URL else f"http://127.0.0.1:{PORT}"

print("\n=== BOT STATUS ===")
print(f"Process: {status}")
print(f"Running URL: {running_url}")
print(f"Health endpoint: {running_url}/health")
print("\nKeep this cell running to continue streaming logs.")

### Notes
- If you do not provide `PUBLIC_BASE_URL`, links will be generated using `http://127.0.0.1:<PORT>`.
- For public access from Colab, use a tunnel service (Cloudflare Tunnel / ngrok) and put that URL in `PUBLIC_BASE_URL`.
- Re-run the last cell to restart the bot process.